# Initialize: attack graph benchmark
Defines the environment, the latent context, the defender's two-part representation, the stage-game solver, the networks, and the training updates used by the other notebooks.

Conventions:
- **Payoff.** `Reward` returns the **defender's** expected payoff after the transition: 0.1 per uncompromised host, plus each valued server that stays uncompromised, minus 0.1 for a direct server-cleaning action. The critic learns the defender's payoff, and the defender is the row maximizer in `MatrixValue`.
- **Latent context.** $\theta = (\theta^{\mathrm{env}}, \theta^{\mathrm{type}})$ is drawn once per episode by `Sample_Context`: the hidden wiring from two terminal hosts to the two servers (unknown to both players), and the attacker's type (which servers it values and its irrationality level $\lambda \sim \mathrm{Unif}[0, 0.5]$), known only to the attacker.
- **Representation** $z_k = [z^S_k, z^\Theta_k]$. The state component $z^S$ is a per-node Bayesian filter of compromise probabilities (`Def_State_Blf`). The context component $z^\Theta$ is a Transformer over the defender's history rows $[z^S_{k-1}, \text{defended nodes}, z^S_k]$ (`TSTransformer`).
- **Attacker's private state** $m^2_k$: running estimates of the host-to-server wiring (`m2_update`).
- **Encoder objective**, Eq. (3): $\mathcal L_{\mathrm{enc}} = \lambda_r \mathcal L_r + \lambda_p \mathcal L_p + \lambda_\Theta \mathcal L_{\mathrm{SI}}$ (`Train_Encoder`).
- **Game step.** The critic is trained by TD regression onto the minimax value of the next stage game, and both actors are distilled toward its equilibrium strategies by cross-entropy (`Train_Critic`, `Train_Player1`, `Train_Player2`). Embeddings are recomputed with the current encoder at every update.

In [ ]:
import os
import copy
import random
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
from scipy.optimize import linprog
from collections import deque
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.nn import TransformerEncoder, TransformerEncoderLayer

torch.backends.mha.set_fastpath_enabled(False)      # keeps the padding-mask path identical in train and eval
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## Attack graph dynamics

In [ ]:
def all_states_bits(n):
    S = 1 << n
    idx = np.arange(S)[:, None]
    return ((idx >> np.arange(n)) & 1).astype(int)
#################################################################################################################
def compromise_prob_next(x, a, u, D, rho, alpha):
    # Probability that each node is compromised after the transition
    x = np.asarray(x, dtype=int)
    a = np.asarray(a, dtype=bool)
    u = np.asarray(u, dtype=bool)
    n = x.size
    p = np.zeros(n)
    for j in range(n):
        attacked = u[j]
        if attacked and j in OR_PREREQ and x[OR_PREREQ[j]] == 0:    # attack needs its prerequisite node
            attacked = False
        if not attacked and D[j]:                                   # propagation along the hidden wiring
            prop = 1.0 - np.prod([1.0 - rho[i, j] * x[i] for i in D[j]])
            if prop > 0.0:
                attacked = True
        if attacked:
            p[j] = alpha[j] if a[j] else 1
        else:
            p[j] = alpha[j] * x[j] if a[j] else float(x[j])
    return p
#################################################################################################################
def transition_matrix(a, u, D, rho, alpha):
    X = all_states_bits(d)
    S = X.shape[0]
    M = np.zeros((S, S), dtype=float)
    for l in range(S):
        eta = compromise_prob_next(X[l], a, u, D, rho, alpha)
        P = np.where(X, eta, 1.0 - eta)
        M[:, l] = P.prod(axis=1)
    return M
#################################################################################################################
def Build_Graph(s1_p, s2_p):
    # Hidden wiring: terminal hosts (3, 4, 6, 7, 8) to servers 9 and 10, edge weights given by s1_p and s2_p
    edges = {(3,9): s1_p[0], (4,9): s1_p[1], (6,9): s1_p[2], (7,9): s1_p[3], (8,9): s1_p[4],
             (3,10):s2_p[0], (4,10):s2_p[1], (6,10):s2_p[2], (7,10):s2_p[3], (8,10):s2_p[4]}
    D = [set() for _ in range(d)]
    rho = np.zeros((d, d))
    for (i, j), p in edges.items():
        rho[i-1, j-1] = p
        D[j-1].add(i-1)
    return D, rho
#################################################################################################################
def Def_Mask(a):
    am = np.zeros(d, dtype=bool)
    for idx in A[a]:
        am[idx-1] = True
    return am
#################################################################################################################
def Att_Vec(u):
    um = np.zeros(d, dtype=bool)
    for idx in U[u]:
        um[idx-1] = True
    return um
#################################################################################################################
def Next_Dist(x, a, u, D, rho):
    x_bits = State_Matrix[x]
    eta = compromise_prob_next(x_bits, Def_Mask(a), Att_Vec(u), D, rho, alpha)
    P = np.where(State_Matrix, eta, 1.0 - eta)
    return P.prod(axis=1)
#################################################################################################################
def Next_State(x, a, u, D, rho):
    dist = Next_Dist(x, a, u, D, rho)
    return random.choices(range(n), weights=dist)[0]

## Benchmark definition: 8 hosts and 2 servers

In [ ]:
d = 10                                    # nodes 1-8 are hosts, 9 and 10 are servers
n = 2**d
U = [[1], [2], [3], [4], [5], [6], [7], [8]]          # attacker strikes one host per round
n_u = 8
A = [[1,3],[2,6],[5,8],[4],[7],[9],[10]]               # defender's group remediation actions
n_a = 7
OR_PREREQ = {2: 0, 5: 1, 7: 4}                         # attacking node j requires node OR_PREREQ[j] (0-indexed)
TERMINALS = [3, 4, 6, 7, 8]                            # candidate parents of the servers (1-indexed)
alpha0 = 0.05                                          # failed-defense probability
alpha = np.full(d, alpha0)
miss_detec = 0.1
false_detec = 0.1
n_observable = 2                                       # hosts reported per round, plus both servers

Len_Theta = 13                                         # theta = [wiring (10), valued servers (2), lambda (1)]
State_Matrix = all_states_bits(d)

Context_length = 30
Embedding_Size = 64


def Sample_Context():
    # Draw the episode's latent context: wiring theta_env and attacker type theta_type
    Theta = np.zeros(Len_Theta)
    Env_Theta = np.zeros(10)
    p = np.random.permutation(TERMINALS)
    s1_parent, s2_parent = p[0], p[1]
    Env_Theta[s1_parent - 1] = 1.0
    Env_Theta[s2_parent - 1] = 1.0
    Type_Theta = np.zeros(3)
    Type_Theta[0] = np.random.randint(0, 2)                  # values server 9
    Type_Theta[1] = np.random.randint(1 - Type_Theta[0], 2)  # values server 10 (at least one server is valued)
    Type_Theta[2] = np.random.uniform(0.0, 0.5)              # irrationality lambda
    Theta[0:10] = Env_Theta
    Theta[10:13] = Type_Theta

    s1_p = np.zeros(5); s1_p[TERMINALS.index(s1_parent)] = 1.0
    s2_p = np.zeros(5); s2_p[TERMINALS.index(s2_parent)] = 1.0
    D, rho = Build_Graph(s1_p, s2_p)
    return Theta, Type_Theta, D, rho

## Stage-game solver

In [ ]:
def MatrixValue(M):
    # Exact minimax by linear programming. Row player maximizes, column player minimizes.
    # Returns (value, row strategy p, column strategy q).
    M = np.asarray(M, dtype=float)
    if M.ndim != 2:
        raise ValueError("M must be a 2D array.")
    m, n = M.shape
    if m == 0 or n == 0 or not np.isfinite(M).all():
        return float(np.nanmean(M)), np.ones(m)/m if m else np.array([]), np.ones(n)/n if n else np.array([])
    # normalization
    center = float(M.mean())
    A = M - center
    scale = float(np.max(np.abs(A)))
    if scale == 0.0:
        return center, np.ones(m)/m, np.ones(n)/n
    A /= scale
    c1 = np.zeros(m + 1); c1[0] = -1.0
    A_ub1 = np.zeros((n + m, m + 1))
    b_ub1 = np.zeros(n + m)
    A_ub1[:n, 0] =  1.0
    A_ub1[:n, 1:] = -A.T
    A_ub1[n:, 1:] = -np.eye(m)
    A_eq1 = np.ones((1, m + 1)); A_eq1[0, 0] = 0.0
    b_eq1 = np.array([1.0])
    opt1 = linprog(c=c1, A_ub=A_ub1, b_ub=b_ub1, A_eq=A_eq1, b_eq=b_eq1, method="highs", options={"presolve": True})
    v_row = None; p = None
    if opt1.success and opt1.x is not None:
        v_row = float(opt1.x[0])
        p = opt1.x[1:]
    c2 = np.zeros(n + 1); c2[0] = 1.0
    A_ub2 = np.zeros((m + n, n + 1))
    b_ub2 = np.zeros(m + n)
    A_ub2[:m, 0]  = -1.0
    A_ub2[:m, 1:] =  A
    A_ub2[m:, 1:] = -np.eye(n)
    A_eq2 = np.ones((1, n + 1)); A_eq2[0, 0] = 0.0
    b_eq2 = np.array([1.0])
    opt2 = linprog(c=c2, A_ub=A_ub2, b_ub=b_ub2, A_eq=A_eq2, b_eq=b_eq2, method="highs", options={"presolve": True})
    v_col = None; q = None
    if opt2.success and opt2.x is not None:
        v_col = float(opt2.x[0])
        q = opt2.x[1:]
    if p is None and q is None:
        return center, np.ones(m)/m, np.ones(n)/n
    if p is None:
        p = np.ones(m)/m
        v_norm = v_col
    elif q is None:
        q = np.ones(n)/n
        v_norm = v_row
    else:
        v_norm = 0.5 * (v_row + v_col)
    p = np.clip(p, 0.0, 1.0); ps = p.sum(); p = p/ps if ps > 0 else np.ones(m)/m
    q = np.clip(q, 0.0, 1.0); qs = q.sum(); q = q/qs if qs > 0 else np.ones(n)/n
    value = center + scale * float(v_norm)
    return value, p, q

## Reward, observations, defender's state filter, attacker's private state

In [ ]:
def Reward(x, a, u, Theta, D, rho):
    # Defender's expected payoff over the next state
    dist = Next_Dist(x, a, u, D, rho)
    expected_next_state = State_Matrix.T @ dist
    target1 = Theta[10]; target2 = Theta[11]
    r = 0.1 * np.sum(1.0 - expected_next_state[:8])
    r += (1.0 - expected_next_state[8]) * target1
    r += (1.0 - expected_next_state[9]) * target2
    if a == 5 or a == 6:                                   # direct server cleaning has a cost
        r -= 0.1
    return r
#################################################################################################################
def Reward2(x, a, u, Theta, next_x):
    # Realized payoff on the valued servers only, given the sampled next state
    next_state = State_Matrix[next_x, :]
    target1 = Theta[10]; target2 = Theta[11]
    if target1 > 0.5 and target2 < 0.5:
        r = (1.0 - next_state[8])
    if target1 < 0.5 and target2 > 0.5:
        r = (1.0 - next_state[9])
    if target1 > 0.5 and target2 > 0.5:
        r = 0.5 * (1.0 - next_state[8]) + 0.5 * (1.0 - next_state[9])
    if a == 5 or a == 6:
        r -= 0.1
    return r
#################################################################################################################
def Observation(x):
    # Noisy alerts: +1 compromised, -1 clean, 0 unreported. Both servers and n_observable random hosts are reported.
    y = np.zeros(d)

    def emit(compromised):
        if compromised:
            return -1 if np.random.rand() < miss_detec else 1
        return 1 if np.random.rand() < false_detec else -1

    hosts = np.random.choice(range(8), size=n_observable, replace=False)
    for i in hosts:
        y[i] = emit(x[i] == 1)
    for j in (8, 9):
        y[j] = emit(x[j] == 1)
    return y
#################################################################################################################
def Def_State_Blf(z_s, y, defended_nodes):
    # State component z^S: per-node compromise probabilities, updated by the defense step and the alert likelihood
    z = z_s.copy()
    for node in defended_nodes:
        z[node - 1] *= alpha0
    for i in range(z.size):
        if y[i] == 1:
            num = (1 - miss_detec) * z[i]
            z[i] =  num / (num + false_detec * (1 - z[i]))
        elif y[i] == -1:
            num = miss_detec * z[i]
            z[i] = num / (num + (1 - false_detec) * (1 - z[i]))
    return np.clip(z, 1e-4, 1 - 1e-4)
#################################################################################################################
def m2_update(s_k, s_next, m2, counts):
    # Attacker's private state m^2: running estimate of each terminal host's link to each server
    for server, hosts in ((8, [2, 3, 5, 6, 7]), (9, [2, 3, 5, 6, 7])):
        if s_k[server] == 1:
            continue
        active = [h for h in hosts if s_k[h] == 1]
        if not active:
            continue
        flipped = (s_next[server] == 1)
        target = 1.0 / len(active) if flipped else 0.0
        for h in active:
            counts[h] += 1
            m2[h] += (target - m2[h]) / counts[h]
    return m2, counts
#################################################################################################################
def Context_Row(z_s, a, next_z_s):
    # One history row for the context encoder: [z^S_{k-1}, defended-node indicator, z^S_k]
    row = np.zeros(d * 3, dtype=np.float32)
    row[0:d] = z_s
    for idx in A[a]:
        row[d + idx - 1] = 1
    row[2*d:3*d] = next_z_s
    return row

def Context_Mask(num_rows):
    # Valid rows of the context window (at least one, so the encoder always has an input)
    mask = torch.zeros((1, Context_length), dtype=torch.bool, device=device)
    mask[0, :max(1, min(num_rows, Context_length))] = True
    return mask

## Replay buffer and networks

In [ ]:
class ReplayMemory(object):
    def __init__(self, capacity):
        self.memory = deque(maxlen=capacity)

    def push(self, args):
        self.memory.append(args)

    def sample(self, batch_size):
        return random.sample(self.memory, batch_size)

    def __len__(self):
        return len(self.memory)
#################################################################################################################
class TSTransformer(nn.Module):
    # Context component z^Theta: Transformer with learned positional embeddings, masked mean pooling, feedforward head
    def __init__(self):
        super(TSTransformer, self).__init__()
        self.input_projection = nn.Linear(d*3, 128)
        self.position_embeddings = nn.Embedding(Context_length, 128)
        encoder_layers = TransformerEncoderLayer(d_model=128, nhead=4, batch_first=True, dropout=0.1)
        self.transformer_encoder = TransformerEncoder(encoder_layers, num_layers=3, enable_nested_tensor=False)
        self.fc1 = nn.Linear(128, 128)
        self.norm1 = nn.LayerNorm(128)
        self.fc2 = nn.Linear(128, Embedding_Size)
        self.activation = nn.ReLU()
        self.dropout = nn.Dropout(0.1)

    def forward(self, x, mask):
        x = self.input_projection(x)
        batch_size, seq_length, _ = x.shape
        position_ids = torch.arange(seq_length, device=x.device).unsqueeze(0).expand(batch_size, -1)
        x = x + self.position_embeddings(position_ids)
        key_padding_mask = ~mask.bool()
        x = self.transformer_encoder(x, src_key_padding_mask=key_padding_mask)
        mask = mask.unsqueeze(-1)
        x = (x * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1e-9)
        x = self.fc1(x)
        x = self.norm1(x)
        x = self.activation(x)
        x = self.dropout(x)
        return self.fc2(x)
#################################################################################################################
class DefenderActor(nn.Module):
    # pi^1(a | z), with z = [z^S, z^Theta]
    def __init__(self):
        super().__init__()
        self.enc_zs1  = nn.Linear(d, 128)
        self.enc_zs2  = nn.Linear(128, 128)
        self.enc_emb1 = nn.Linear(Embedding_Size, 128)
        self.enc_emb2 = nn.Linear(128, 128)
        self.fuse1 = nn.Linear(256, 256)
        self.ln    = nn.LayerNorm(256)
        self.fuse2 = nn.Linear(256, 128)
        self.head  = nn.Linear(128, n_a)

    def forward(self, z_s, Embeding):
        zb = F.relu(self.enc_zs2(F.relu(self.enc_zs1(z_s))))
        zc = F.relu(self.enc_emb2(F.relu(self.enc_emb1(Embeding))))
        h = torch.cat([zb, zc], dim=-1)
        h = self.ln(F.relu(self.fuse1(h)))
        h = F.relu(self.fuse2(h))
        return F.softmax(self.head(h), dim=-1)
#################################################################################################################
class AttackerActor(nn.Module):
    # beta(u | state, m^2, theta_type): the physical attacker, acting on its own private information
    def __init__(self):
        super().__init__()
        self.enc_x1   = nn.Linear(d, 128)
        self.enc_x2   = nn.Linear(128, 128)
        self.enc_m2_1 = nn.Linear(d, 128)
        self.enc_m2_2 = nn.Linear(128, 128)
        self.enc_ty1  = nn.Linear(3, 64)
        self.enc_ty2  = nn.Linear(64, 128)
        self.fuse1 = nn.Linear(384, 256)
        self.ln    = nn.LayerNorm(256)
        self.fuse2 = nn.Linear(256, 128)
        self.head  = nn.Linear(128, n_u)

    def forward(self, x, m2, type_theta):
        zx = F.relu(self.enc_x2(F.relu(self.enc_x1(x))))
        zm = F.relu(self.enc_m2_2(F.relu(self.enc_m2_1(m2))))
        zt = F.relu(self.enc_ty2(F.relu(self.enc_ty1(type_theta))))
        h = torch.cat([zx, zm, zt], dim=-1)
        h = self.ln(F.relu(self.fuse1(h)))
        h = F.relu(self.fuse2(h))
        return F.softmax(self.head(h), dim=-1)
#################################################################################################################
class Critic(nn.Module):
    # Centralized Q(z, state, m^2, theta) over all joint actions, the defender's payoff
    def __init__(self):
        super().__init__()
        self.x_dim     = d
        self.theta_dim = Len_Theta
        self.embd_dim  = Embedding_Size

        self.enc_zs1    = nn.Linear(self.x_dim, 128)
        self.enc_zs2    = nn.Linear(128, 128)
        self.enc_embd1  = nn.Linear(self.embd_dim, 128)
        self.enc_embd2  = nn.Linear(128, 128)
        self.enc_state1 = nn.Linear(self.x_dim, 128)
        self.enc_state2 = nn.Linear(128, 128)
        self.enc_m2_1   = nn.Linear(self.x_dim, 128)
        self.enc_m2_2   = nn.Linear(128, 128)
        self.enc_type1  = nn.Linear(3, 64)
        self.enc_type2  = nn.Linear(64, 128)
        self.enc_theta1 = nn.Linear(self.theta_dim, 64)
        self.enc_theta2 = nn.Linear(64, 128)

        self.ln_fuse = nn.LayerNorm(128 * 6)
        self.fuse1   = nn.Linear(128 * 6, 512)
        self.fuse2   = nn.Linear(512, 128)
        self.out     = nn.Linear(128, n_a * n_u)

    def forward(self, z_s, Embeding, x, m2, type_theta, theta):
        z_b = F.relu(self.enc_zs2(F.relu(self.enc_zs1(z_s))))
        z_c = F.relu(self.enc_embd2(F.relu(self.enc_embd1(Embeding))))
        z_x = F.relu(self.enc_state2(F.relu(self.enc_state1(x))))
        z_m = F.relu(self.enc_m2_2(F.relu(self.enc_m2_1(m2))))
        z_y = F.relu(self.enc_type2(F.relu(self.enc_type1(type_theta))))
        z_t = F.relu(self.enc_theta2(F.relu(self.enc_theta1(theta))))
        e = torch.cat([z_b, z_c, z_x, z_m, z_y, z_t], dim=-1)
        e = self.ln_fuse(e)
        e = F.relu(self.fuse1(e))
        e = F.relu(self.fuse2(e))
        return self.out(e)

## Representation step: encoder objective, Eq. (3)
$\mathcal L_{\mathrm{enc}} = \lambda_r \mathcal L_r + \lambda_p \mathcal L_p + \lambda_\Theta \mathcal L_{\mathrm{SI}}$:
- $\mathcal L_r$: regression of the stage reward on the representation, through a reward head.
- $\mathcal L_p$: prediction of the next representation from $(z_k, a^1_k, y_{k+1})$ through a latent transition model and a prediction head, against gradient-free exponential-moving-average copies of the encoder and projection. `F.mse_loss` averages over the 128 projection dimensions, so it is the squared norm in Eq. (3) divided by 128; the constant is absorbed into $\lambda_p$.
- $\mathcal L_{\mathrm{SI}}$: Soft-InfoNCE on the $\ell_2$-normalized projection, $\mathcal L_i = -\sum_{j\ne i} w_{ij}\log p_{ij}$, with $p_{ij}$ a softmax of $g_i^\top g_j/T$ over $k \ne i$ and graded targets $w_{ij} = s_{ij}/\sum_{\ell\ne i}s_{i\ell}$, $s_{ij} = \exp(-\tilde d_\Theta(\theta_i,\theta_j)/\sigma_\Theta)$.

The computable proxy is $\tilde d_\Theta = d_{\mathrm{tgt}} + d_{\mathrm{wire}} + d_\lambda$ with $\sigma_\Theta = 1$:
- $d_{\mathrm{tgt}} = -\log J(\mathrm{tgt}_i, \mathrm{tgt}_j)$, where $J$ is the Jaccard similarity of the valued-server sets (infinite when disjoint, so $s_{ij} = 0$),
- $d_{\mathrm{wire}} = -\log\big(1 - g_{ij}\,\lVert\rho_i - \rho_j\rVert_1/4\big)$, with gate $g_{ij} = 1 - \max(\lambda_i, \lambda_j)$, since the wiring matters less against an irrational attacker,
- $d_\lambda = \lvert\lambda_i - \lambda_j\rvert / 0.25$.

So $s_{ij}$ is the product of the three component similarities, as computed in `Context_Similarity`.

In [ ]:
def Context_Similarity(th):
    # s_ij = exp(-d_tilde(theta_i, theta_j)), written as the product of the three component similarities
    rho = th[:, 0:10]
    tgt = th[:, 10:12]
    lam = th[:, 12]
    inter = torch.minimum(tgt.unsqueeze(1), tgt.unsqueeze(0)).sum(-1)
    union = torch.maximum(tgt.unsqueeze(1), tgt.unsqueeze(0)).sum(-1)
    tgt_sim = torch.where(union > 0, inter / union, torch.ones_like(union))
    wire_sim = 1.0 - torch.cdist(rho, rho, p=1) / 4.0
    gate = 1.0 - torch.maximum(lam.unsqueeze(1), lam.unsqueeze(0))
    wire_sim = 1.0 - gate * (1.0 - wire_sim)
    lam_sim = torch.exp(-torch.abs(lam.unsqueeze(1) - lam.unsqueeze(0)) / 0.25)
    return tgt_sim * wire_sim * lam_sim
#################################################################################################################
def ema_update(tau=0.995):
    # Gradient-free targets for the next-representation loss
    for po, pt in zip(Context_Model.parameters(), target_model.parameters()):
        pt.data.lerp_(po.data, 1.0 - tau)
    for po, pt in zip(proj_online.parameters(), proj_target.parameters()):
        pt.data.lerp_(po.data, 1.0 - tau)
    for bo, bt in zip(Context_Model.buffers(), target_model.buffers()):
        bt.data.copy_(bo.data)
    for bo, bt in zip(proj_online.buffers(), proj_target.buffers()):
        bt.data.copy_(bo.data)
#################################################################################################################
def Train_Encoder():
    if len(context_memory) < CONTEXT_BATCH:
        return 0.0
    batch = context_memory.sample(CONTEXT_BATCH)
    (ctx, msk, nctx, nmsk, a, y, r, th) = map(list, zip(*batch))
    ctx  = torch.cat(ctx,  dim=0).to(device)
    msk  = torch.cat(msk,  dim=0).to(device)
    nctx = torch.cat(nctx, dim=0).to(device)
    nmsk = torch.cat(nmsk, dim=0).to(device)
    a  = torch.stack(a).to(device)
    y  = torch.stack(y).to(device).float()
    r  = torch.stack(r).to(device).float()
    th = torch.stack(th).to(device).float()
    B = ctx.shape[0]

    Context_Model.train(); trans_pred.train(); reward_head.train()
    proj_online.train();  pred_online.train()
    z = Context_Model(ctx, msk)

    # L_r: reward prediction
    loss_r = F.mse_loss(reward_head(z).squeeze(-1), r)

    # L_p: next-representation prediction against the EMA target encoder and projection
    z_hat = trans_pred(torch.cat([z, F.one_hot(a, num_classes=n_a).float(), y], dim=-1))
    h_hat = F.normalize(pred_online(proj_online(z_hat)), p=2, dim=1)
    with torch.no_grad():
        h_tgt = F.normalize(proj_target(target_model(nctx, nmsk)), p=2, dim=1)
    loss_p = F.mse_loss(h_hat, h_tgt)

    # L_SI: Soft-InfoNCE with graded targets from the context similarity
    g = F.normalize(proj_online(z), p=2, dim=1)
    eye = torch.eye(B, device=device, dtype=torch.bool)
    logits = ((g @ g.t()) / SI_TEMPERATURE).masked_fill(eye, -1e9)
    s = Context_Similarity(th).masked_fill(eye, 0.0)
    w = s / s.sum(dim=1, keepdim=True).clamp(min=1e-8)
    loss_si = -(w * F.log_softmax(logits, dim=1)).sum(dim=1).mean()

    loss = LAMBDA_R * loss_r + LAMBDA_P * loss_p + LAMBDA_THETA * loss_si

    encoder_optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(
        list(Context_Model.parameters()) + list(trans_pred.parameters()) +
        list(reward_head.parameters())  + list(proj_online.parameters()) +
        list(pred_online.parameters()), max_norm=1.0)
    encoder_optimizer.step()
    ema_update()

    Context_Model.eval()
    return loss.item()

## Game step: critic regression and actor distillation

In [ ]:
def _game_batch(batch_size):
    # Sample transitions and recompute the context embeddings with the current encoder
    batch = exp_memory.sample(batch_size)
    (x, nx, zs, nzs, m2, nm2, a, u, r, tt, th, ctx, msk, nctx, nmsk) = map(list, zip(*batch))
    S = lambda v: torch.stack(v).to(device)
    C = lambda v: torch.cat(v, dim=0).to(device)
    Context_Model.eval()
    with torch.no_grad():
        emb  = Context_Model(C(ctx), C(msk))
        nemb = Context_Model(C(nctx), C(nmsk))
    return dict(x=S(x), nx=S(nx), zs=S(zs), nzs=S(nzs), m2=S(m2), nm2=S(nm2), a=S(a), u=S(u),
                r=S(r), tt=S(tt), th=S(th), emb=emb, nemb=nemb)
#################################################################################################################
def Train_Critic():
    if len(exp_memory) < CRITIC_BATCH:
        return 0.0
    b = _game_batch(CRITIC_BATCH)
    Bsz = b['x'].shape[0]

    q_flat   = critic(b['zs'], b['emb'], b['x'], b['m2'], b['tt'], b['th'])
    q_values = q_flat.gather(1, (b['a'] * n_u + b['u']).unsqueeze(1)).squeeze(1)

    with torch.no_grad():
        next_q = critic_target(b['nzs'], b['nemb'], b['nx'], b['nm2'], b['tt'], b['th']).view(Bsz, n_a, n_u).cpu().numpy()
        V_next = torch.tensor([MatrixValue(next_q[i])[0] for i in range(Bsz)], dtype=torch.float32, device=device)
        target_values = b['r'] + GAMMA * V_next

    loss = F.smooth_l1_loss(q_values, target_values)
    critic_optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(critic.parameters(), max_norm=1.0)
    critic_optimizer.step()

    with torch.no_grad():
        for tgt, src in zip(critic_target.parameters(), critic.parameters()):
            tgt.data.mul_(1.0 - TAU).add_(TAU * src.data)
    return loss.item()
#################################################################################################################
def Train_Player1():
    # Defender: cross-entropy toward the row (maximizing) equilibrium strategy of the critic's stage game
    if len(exp_memory) < PLAYER1_BATCH:
        return 0.0
    b = _game_batch(PLAYER1_BATCH)
    Bsz = b['x'].shape[0]
    with torch.no_grad():
        Q = critic(b['zs'], b['emb'], b['x'], b['m2'], b['tt'], b['th']).view(Bsz, n_a, n_u).cpu().numpy()
        tgt_a = torch.tensor(np.stack([MatrixValue(Q[i])[1] for i in range(Bsz)]), dtype=torch.float32, device=device)

    pred_a = Player1(b['zs'], b['emb'])
    loss = -(tgt_a * torch.log(pred_a + 1e-8)).sum(dim=1).mean()

    player1_optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(Player1.parameters(), max_norm=1.0)
    player1_optimizer.step()
    return loss.item()
#################################################################################################################
def Train_Player2():
    # Attacker: cross-entropy toward the column (minimizing) equilibrium strategy, acting on its private information
    if len(exp_memory) < PLAYER2_BATCH:
        return 0.0
    b = _game_batch(PLAYER2_BATCH)
    Bsz = b['x'].shape[0]
    with torch.no_grad():
        Q = critic(b['zs'], b['emb'], b['x'], b['m2'], b['tt'], b['th']).view(Bsz, n_a, n_u).cpu().numpy()
        tgt_u = torch.tensor(np.stack([MatrixValue(Q[i])[2] for i in range(Bsz)]), dtype=torch.float32, device=device)

    pred_u = Player2(b['x'], b['m2'], b['tt'])
    loss = -(tgt_u * torch.log(pred_u + 1e-8)).sum(dim=1).mean()

    player2_optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(Player2.parameters(), max_norm=1.0)
    player2_optimizer.step()
    return loss.item()

## Baseline architectures and the best-response attacker (shared by baselines, best_response, and eval)

In [ ]:
OBS_DIM = d + d + n_a          # baseline observation: [z_s, y, a_prev one-hot]
RNN_H = 128
LATENT_DIM = 16


def sample_config():
    # Episode context and initial state, drawn in the same order as main.ipynb
    Theta, Type_Theta, D, rho = Sample_Context()
    x0 = np.random.randint(n)
    return Theta, Type_Theta, float(Type_Theta[2]), x0, D, rho
#################################################################################################################
class RecurrentAC(nn.Module):
    # Recurrent actor-critic over the baseline observation, optionally conditioned on an extra input
    def __init__(self, extra_dim=0, rnn_h=RNN_H):
        super().__init__()
        self.extra_dim = extra_dim
        self.gru = nn.GRU(OBS_DIM, rnn_h, batch_first=True)
        dd = rnn_h + extra_dim
        self.pi = nn.Sequential(nn.Linear(dd, 128), nn.ReLU(), nn.Linear(128, n_a))
        self.v  = nn.Sequential(nn.Linear(dd, 128), nn.ReLU(), nn.Linear(128, 1))
    def forward(self, obs, extra=None, h0=None):
        out, hT = self.gru(obs, h0)
        feat = out if extra is None else torch.cat([out, extra], dim=-1)
        return self.pi(feat), self.v(feat).squeeze(-1), hT
#################################################################################################################
class VariBADEncoder(nn.Module):
    def __init__(self, latent_dim=LATENT_DIM, rnn_h=RNN_H):
        super().__init__()
        self.gru = nn.GRU(OBS_DIM, rnn_h, batch_first=True)
        self.mu = nn.Linear(rnn_h, latent_dim)
        self.logvar = nn.Linear(rnn_h, latent_dim)
    def forward(self, obs, h0=None):
        out, hT = self.gru(obs, h0)
        return self.mu(out), self.logvar(out), hT
#################################################################################################################
class PEARLEncoder(nn.Module):
    # Product-of-Gaussians posterior over the latent from the episode's transitions (y, a, y_next)
    def __init__(self, latent_dim=LATENT_DIM):
        super().__init__()
        self.latent_dim = latent_dim
        self.net = nn.Sequential(nn.Linear(d + n_a + d, 128), nn.ReLU(), nn.Linear(128, 2 * latent_dim))
    def factors(self, trans):
        out = self.net(trans)
        mu = out[..., :self.latent_dim]
        var = F.softplus(out[..., self.latent_dim:]) + 1e-6
        return mu, var
    def posterior(self, trans):
        if trans.shape[0] == 0:
            return torch.zeros(self.latent_dim, device=device), torch.ones(self.latent_dim, device=device)
        mu_i, var_i = self.factors(trans)
        prec = 1.0 / var_i
        prec_post = 1.0 + prec.sum(0)
        mu_post = (prec * mu_i).sum(0) / prec_post
        return mu_post, 1.0 / prec_post
#################################################################################################################
class BRAttacker(nn.Module):
    # Recurrent best-response attacker: GRU over [true state, previous action one-hot], conditioned on the full theta
    def __init__(self, rnn_h=RNN_H):
        super().__init__()
        self.gru = nn.GRU(d + n_u, rnn_h, batch_first=True)
        self.theta_enc = nn.Sequential(nn.Linear(Len_Theta, 64), nn.ReLU())
        dd = rnn_h + 64
        self.pi = nn.Sequential(nn.Linear(dd, 128), nn.ReLU(), nn.Linear(128, n_u))
        self.v  = nn.Sequential(nn.Linear(dd, 128), nn.ReLU(), nn.Linear(128, 1))
    def forward(self, obs, theta, h0=None):
        out, hT = self.gru(obs, h0)
        feat = torch.cat([out, self.theta_enc(theta)], dim=-1)
        return self.pi(feat), self.v(feat).squeeze(-1), hT

## Loaders and defender wrappers (shared by best_response, eval, and compromise)

In [ ]:
def Load(net, path):
    net.load_state_dict(torch.load(path, map_location=device, weights_only=True))
    return net.eval()


def Load_Baselines(save_dir="compared"):
    # The five trained baselines from baselines.ipynb
    ck_ = lambda name: torch.load(os.path.join(save_dir, name + ".pth"), map_location=device, weights_only=True)
    out = {}
    for name in ('RPPO', 'DR_RPPO'):
        ck = ck_(name)
        pol = RecurrentAC(0, ck['config']['RNN_H']).to(device); pol.load_state_dict(ck['policy']); pol.eval()
        out[name] = {'policy': pol}
    ck = ck_("VARIBAD"); L = ck['config']['LATENT_DIM']
    pol = RecurrentAC(2 * L, ck['config']['RNN_H']).to(device); pol.load_state_dict(ck['policy']); pol.eval()
    enc = VariBADEncoder(L, ck['config']['RNN_H']).to(device); enc.load_state_dict(ck['encoder']); enc.eval()
    out['VARIBAD'] = {'policy': pol, 'encoder': enc, 'latent_dim': L}
    ck = ck_("PEARL"); L = ck['config']['LATENT_DIM']
    pol = RecurrentAC(L, ck['config']['RNN_H']).to(device); pol.load_state_dict(ck['policy']); pol.eval()
    enc = PEARLEncoder(L).to(device); enc.load_state_dict(ck['encoder']); enc.eval()
    out['PEARL'] = {'policy': pol, 'encoder': enc, 'latent_dim': L, 'resample_k': ck['config'].get('resample_k', 10)}
    ck = ck_("MMD")
    pol = RecurrentAC(0, ck['config']['RNN_H']).to(device); pol.load_state_dict(ck['p1']); pol.eval()
    out['MMD'] = {'policy': pol}
    return out
#################################################################################################################
# Defender wrappers: reset(...) at the start of an episode, act_dist(x_bits, m2) for the action distribution,
# observe(a, x_bits, next_x_bits, y) after each transition. They use the globals Context_Model, critic, Player1,
# and BASELINES, which each evaluation notebook loads.
class PipelineDefender:
    # 'Trained P1': our defender on [z_s, context embedding]. 'Critic P1': the Oracle, the critic's equilibrium row
    # with privileged access to the state and context.
    def __init__(self, name):
        self.name = name
    def reset(self, Theta, Type_Theta, lam, x0, D, rho):
        self.th = torch.tensor(Theta, dtype=torch.float32, device=device)
        self.tt = torch.tensor(Type_Theta, dtype=torch.float32, device=device)
        self.z_s = np.ones(d) / 2
        self.Context = np.zeros([Context_length, d * 3], dtype=np.float32)
        self.m2 = np.full(d, 0.2)
        self.counts = np.zeros(d)
        self.t = 0
    def act_dist(self, x_bits, m2):
        ctx = torch.tensor(self.Context, dtype=torch.float32, device=device).view(1, Context_length, d * 3)
        with torch.no_grad():
            Emb = Context_Model(ctx, Context_Mask(self.t)).squeeze(0)
        z_s_t = torch.tensor(self.z_s, dtype=torch.float32, device=device)
        if self.name == 'Trained P1':
            with torch.no_grad():
                p = Player1(z_s_t, Emb).cpu().numpy()
        else:
            x_t = torch.tensor(x_bits, dtype=torch.float32, device=device)
            m2_t = torch.tensor(self.m2, dtype=torch.float32, device=device)
            with torch.no_grad():
                Q = critic(z_s_t, Emb, x_t, m2_t, self.tt, self.th).cpu().numpy().reshape(n_a, n_u)
            _, p, _ = MatrixValue(Q)
        p = np.clip(np.asarray(p, dtype=np.float64), 0, None)
        return p / p.sum()
    def observe(self, a, x_bits, next_x_bits, y):
        if self.name == 'Critic P1':
            self.m2, self.counts = m2_update(x_bits, next_x_bits, self.m2, self.counts)
        next_z_s = Def_State_Blf(self.z_s, y, A[a])
        row = Context_Row(self.z_s, a, next_z_s)
        if self.t < Context_length:
            self.Context[self.t, :] = row
        else:
            self.Context[:-1, :] = self.Context[1:, :]
            self.Context[-1, :] = row
        self.z_s = next_z_s
        self.t += 1


class RandomDefender:
    def reset(self, Theta, Type_Theta, lam, x0, D, rho):
        pass
    def act_dist(self, x_bits, m2):
        return np.full(n_a, 1.0 / n_a)
    def observe(self, a, x_bits, next_x_bits, y):
        pass


class BaselineDefender:
    # A trained baseline acting on [z_s, y, a_prev], with VariBAD's belief or PEARL's sampled latent as extra input
    def __init__(self, name):
        self.name = name
        self.m = BASELINES[name]
    def reset(self, Theta, Type_Theta, lam, x0, D, rho):
        self.h = None
        self.z_s = np.ones(d) / 2
        self.y_cur = np.zeros(d)
        self.a_prev = np.zeros(n_a)
        if self.name == 'VARIBAD':
            self.enc_h = None
        if self.name == 'PEARL':
            self.ctx = []
            self.z = np.random.randn(self.m['latent_dim']).astype(np.float32)
    def _extra(self, obs):
        if self.name == 'VARIBAD':
            with torch.no_grad():
                mu, logvar, self.enc_h = self.m['encoder'](
                    torch.tensor(obs, dtype=torch.float32, device=device).view(1, 1, -1), self.enc_h)
            mu = mu.view(-1).cpu().numpy()
            std = np.exp(0.5 * logvar.view(-1).cpu().numpy())
            return np.concatenate([mu, std])
        if self.name == 'PEARL':
            return self.z
        return None
    def act_dist(self, x_bits, m2):
        obs = np.concatenate([self.z_s, self.y_cur, self.a_prev]).astype(np.float32)
        ex_np = self._extra(obs)
        obs_t = torch.tensor(obs, device=device).view(1, 1, OBS_DIM)
        ex_t = None if ex_np is None else torch.tensor(np.asarray(ex_np, dtype=np.float32), device=device).view(1, 1, -1)
        with torch.no_grad():
            logits, _, self.h = self.m['policy'](obs_t, ex_t, self.h)
        p = F.softmax(logits.view(-1), dim=-1).cpu().numpy().astype(np.float64)
        return p / p.sum()
    def observe(self, a, x_bits, next_x_bits, y):
        a_onehot = np.zeros(n_a, dtype=np.float32); a_onehot[a] = 1.0
        if self.name == 'PEARL':
            self.ctx.append(np.concatenate([self.y_cur, a_onehot, y]).astype(np.float32))
            if len(self.ctx) % self.m['resample_k'] == 0:
                with torch.no_grad():
                    mu, var = self.m['encoder'].posterior(torch.tensor(np.stack(self.ctx), device=device))
                self.z = (mu + var.sqrt() * torch.randn_like(mu)).cpu().numpy()
        self.z_s = Def_State_Blf(self.z_s, y, A[a])
        self.y_cur = y
        self.a_prev = a_onehot


def get_defender(name):
    if name in ('Critic P1', 'Trained P1'):
        return PipelineDefender(name)
    if name == 'Random P1':
        return RandomDefender()
    return BaselineDefender(name)


def Context_From_Config(s1_parent, s2_parent, t0, t1, lam):
    # Rebuild an episode's context from a pre-drawn configuration
    Theta = np.zeros(Len_Theta)
    Theta[s1_parent - 1] = 1.0
    Theta[s2_parent - 1] = 1.0
    Type_Theta = np.array([t0, t1, lam], dtype=float)
    Theta[10:13] = Type_Theta
    s1_p = np.zeros(5); s1_p[TERMINALS.index(s1_parent)] = 1.0
    s2_p = np.zeros(5); s2_p[TERMINALS.index(s2_parent)] = 1.0
    D, rho = Build_Graph(s1_p, s2_p)
    return Theta, Type_Theta, D, rho